# EMCAD Experiment 2 — BraTS Run 1
## Part 5 FINAL v3: Epochs 151–200 + Full Held-Out Test


In [1]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import tarfile
import sys

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

EXPECTED_COMMIT = "26c9c31f731f749b62c5fe83f44376dac75f3aa8"

def sha256(path):
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()

def select_unique(filename, label):
    files = sorted(
        set(INPUT.rglob(filename))
    )

    if not files:
        raise FileNotFoundError(
            f"{label} not found: {filename}\n"
            "Check that the correct saved Kaggle notebook output is attached."
        )

    if len(files) == 1:
        return files[0]

    groups = {}

    for path in files:
        groups.setdefault(
            sha256(path),
            [],
        ).append(path)

    if len(groups) == 1:
        print(
            f"{label}: {len(files)} identical copies found; using first."
        )
        return files[0]

    raise RuntimeError(
        f"{label}: conflicting copies found:\n"
        + "\n".join(str(p) for p in files)
    )

def expected_hash(manifest, filename):
    for line in manifest.read_text().splitlines():
        parts = line.strip().split()

        if (
            len(parts) >= 2
            and parts[-1] == filename
        ):
            return parts[0]

    raise RuntimeError(
        f"{filename} was not found in {manifest.name}"
    )

print("System Python:", sys.version.split()[0])
print("Kaggle input roots:")
for path in sorted(INPUT.iterdir()):
    print(" -", path.name)


System Python: 3.13.15
Kaggle input roots:
 - notebooks


In [2]:
# Locate ONLY the two required inputs.

brats_archive = select_unique(
    "EMCAD_BraTS_Part1_Output.tar.gz",
    "BraTS Part 1 prepared archive",
)

brats_checksums = select_unique(
    "EMCAD_BraTS_Part1_SHA256SUMS.txt",
    "BraTS Part 1 checksum manifest",
)

part4_archive = select_unique(
    "EMCAD_BraTS_Run1_Part4_State.tar.gz",
    "BraTS Run 1 Part 4 state archive",
)

part4_checksums = select_unique(
    "EMCAD_BraTS_Run1_Part4_SHA256SUMS.txt",
    "BraTS Run 1 Part 4 checksum manifest",
)

print("BraTS Part 1:", brats_archive.parent)
print("Part 4:", part4_archive.parent)

assert sha256(brats_archive) == expected_hash(
    brats_checksums,
    brats_archive.name,
)

assert sha256(part4_archive) == expected_hash(
    part4_checksums,
    part4_archive.name,
)

print("BraTS Part 1 archive checksum: PASS")
print("Part 4 state archive checksum: PASS")


BraTS Part 1: /kaggle/input/notebooks/rinkyranisaha/emcad-experiment-2-brats-preparation
Part 4: /kaggle/input/notebooks/rinkyranisaha/part-4-training-epochs-101-150
BraTS Part 1 archive checksum: PASS
Part 4 state archive checksum: PASS


## Rebuild the exact legacy runtime without Notebook 1

This stage uses a clean Python 3.8 environment and the same key package versions used in the earlier successful EMCAD runs:

- Python 3.8
- PyTorch 1.11.0 + CUDA 11.3
- torchvision 0.12.0
- torchaudio 0.11.0
- NumPy 1.22.4
- timm 0.6.12
- mmcv-full 1.7.2

The repository is cloned from the official EMCAD GitHub repository and checked out at the exact pinned commit.


In [3]:
ENV = WORK / "emcadenv"
MAMBA_ROOT = WORK / "mamba_root"
MICROMAMBA = WORK / "bin" / "micromamba"
REPO = WORK / "EMCAD"

for path in [ENV, MAMBA_ROOT, REPO]:
    if path.exists():
        shutil.rmtree(path)

(WORK / "bin").mkdir(
    parents=True,
    exist_ok=True,
)

# Download micromamba as a small self-contained environment manager.
subprocess.run(
    [
        "bash",
        "-lc",
        (
            "set -euo pipefail; "
            "curl -L --fail --retry 8 --retry-delay 3 "
            "https://micro.mamba.pm/api/micromamba/linux-64/latest "
            "| tar -xj -C /kaggle/working bin/micromamba"
        ),
    ],
    check=True,
)

assert MICROMAMBA.is_file()

env_vars = os.environ.copy()
env_vars["MAMBA_ROOT_PREFIX"] = str(MAMBA_ROOT)

subprocess.run(
    [
        str(MICROMAMBA),
        "create",
        "-y",
        "-p",
        str(ENV),
        "-c",
        "conda-forge",
        "python=3.8.20",
        "pip=24.3.1",
    ],
    env=env_vars,
    check=True,
)

PYTHON = ENV / "bin" / "python"
PIP = ENV / "bin" / "pip"

assert PYTHON.is_file()
assert PIP.is_file()

print("Python 3.8 environment created:", ENV)


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  4039    0  4039    0     0   4794      0 --:--:-- --:--:-- --:--:--  4794
100 6824k  100 6824k    0     0  3519k      0  0:00:01  0:00:01 --:--:-- 7268k


Fetch Shard Index for conda-forge/linux-64                                                      ⧖ Starting
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.1 sec)
Fetch Shard Index for conda-forge/noarch                                                        ⧖ Starting
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.1 sec)
Fetching and Parsing Packages' Shards                                                           ⧖ Starting
Fetching and Parsing Packages' Shards                                                     ✔ Done (3.4 sec)

Resolving Environment                                                                           ⧖ Starting
Resolving Environment                                                                     ✔ Done (0.2 sec)

Transaction

  Prefix: /kaggle/working/emcadenv

  Updating specs:

   - python=3.8.20
   - pip=24.3.1




warning  libmamba Security Warning: This transaction includes executing package scripts (pre/post-link/unlink) if present. These scripts can contain arbitrary code. Please ensure you trust the package sources.


  Package               Version  Build                 Channel          Size
──────────────────────────────────────────────────────────────────────────────
  Install:
──────────────────────────────────────────────────────────────────────────────

  + _openmp_mutex           4.5  20_gnu                conda-forge      29kB
  + bzip2                 1.0.8  hda65f42_10           conda-forge     258kB
  + ca-certificates   2026.7.22  hbd8a1cb_0            conda-forge     132kB
  + ld_impl_linux-64     2.46.1  default_hbd61a6d_102  conda-forge     745kB
  + libffi                3.7.0  h81df57d_1            conda-forge      68kB
  + libgcc               16.2.0  ha9f2e26_7            conda-forge       1MB
  + libgomp              16.2.0  he0feb66_7            conda-forge     641kB
  + liblzma               5.8.3  hb03c661_1            conda-forge     113kB
  + liblzma-devel         5.8.3  hb03c661_1            conda-forge     492kB
  + libnsl                2.0.1  hb9d3cd8_1            conda

In [4]:
# Install the same core framework versions used in the previous stages.

subprocess.run(
    [
        str(PIP),
        "install",
        "--no-cache-dir",
        "torch==1.11.0+cu113",
        "torchvision==0.12.0+cu113",
        "torchaudio==0.11.0",
        "--extra-index-url",
        "https://download.pytorch.org/whl/cu113",
    ],
    check=True,
)

subprocess.run(
    [
        str(PIP),
        "install",
        "--no-cache-dir",
        "mmcv-full==1.7.2",
        "-f",
        "https://download.openmmlab.com/mmcv/dist/cu113/torch1.11.0/index.html",
    ],
    check=True,
)

# Minimal deterministic dependency set needed by EMCAD + this BraTS pipeline.
subprocess.run(
    [
        str(PIP),
        "install",
        "--no-cache-dir",
        "numpy==1.22.4",
        "timm==0.6.12",
        "scipy==1.10.1",
        "nibabel==5.2.1",
        "Pillow==10.4.0",
        "einops==0.8.1",
        "thop",
        "ptflops==0.7.2.2",
        "PyYAML",
    ],
    check=True,
)

print("Core EMCAD runtime installation: PASS")


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu113
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 GB 95.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 78.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 213.1 MB/s eta 0:00:00
Looking in links: https://download.openmmlab.com/mmcv/dist/cu113/torch1.11.0/index.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 244.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.0/806.0 kB 681.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 197.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
# Clone the official author repository and pin it to the exact commit used
# in Experiments 1 and BraTS Parts 2–4.

subprocess.run(
    [
        "git",
        "clone",
        "--no-checkout",
        "https://github.com/SLDGroup/EMCAD.git",
        str(REPO),
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "checkout",
        "--detach",
        EXPECTED_COMMIT,
    ],
    check=True,
)

commit = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

assert commit == EXPECTED_COMMIT

subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "diff",
        "--exit-code",
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "diff",
        "--cached",
        "--exit-code",
    ],
    check=True,
)

print("Pinned EMCAD commit:", commit)
print("Tracked author source: CLEAN")


Cloning into '/kaggle/working/EMCAD'...


Pinned EMCAD commit: 26c9c31f731f749b62c5fe83f44376dac75f3aa8
Tracked author source: CLEAN


HEAD is now at 26c9c31 adding trained weights on Synapse dataset


In [6]:
# Verify reconstructed runtime BEFORE spending GPU time.

subprocess.run(
    [
        str(PYTHON),
        "-c",
        (
            "import sys, torch, torchvision, numpy, timm, nibabel, scipy, mmcv; "
            "print('Python:', sys.version.split()[0]); "
            "print('PyTorch:', torch.__version__); "
            "print('Torchvision:', torchvision.__version__); "
            "print('CUDA build:', torch.version.cuda); "
            "print('CUDA available:', torch.cuda.is_available()); "
            "print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'); "
            "print('NumPy:', numpy.__version__); "
            "print('timm:', timm.__version__); "
            "print('mmcv:', mmcv.__version__); "
            "print('Nibabel:', nibabel.__version__); "
            "print('SciPy:', scipy.__version__); "
            "assert sys.version_info[:2] == (3, 8); "
            "assert torch.__version__.startswith('1.11.0'); "
            "assert torchvision.__version__.startswith('0.12.0'); "
            "assert numpy.__version__ == '1.22.4'; "
            "assert timm.__version__ == '0.6.12'; "
            "assert mmcv.__version__ == '1.7.2'; "
            "assert torch.cuda.is_available()"
        ),
    ],
    check=True,
)

print("Reconstructed environment preflight: PASS")


/kaggle/working/emcadenv/lib/python3.8/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(


Python: 3.8.20
PyTorch: 1.11.0+cu113
Torchvision: 0.12.0+cu113
CUDA build: 11.3
CUDA available: True
GPU: Tesla T4
NumPy: 1.22.4
timm: 0.6.12
mmcv: 1.7.2
Nibabel: 5.2.1
SciPy: 1.10.1
Reconstructed environment preflight: PASS


In [7]:
# Restore BraTS prepared data and the cumulative epoch-150 state.

BRATS = WORK / "EMCAD_BraTS_Part1_Output"
PART4 = WORK / "EMCAD_BraTS_Run1_Part4_State"

for path in [BRATS, PART4]:
    if path.exists():
        shutil.rmtree(path)

with tarfile.open(
    brats_archive,
    "r:gz",
) as tf:
    tf.extractall(WORK)

with tarfile.open(
    part4_archive,
    "r:gz",
) as tf:
    tf.extractall(WORK)

assert BRATS.is_dir()
assert PART4.is_dir()

train_cache = sorted(
    (BRATS / "slice_cache" / "train").glob("*.pt")
)

val_cache = sorted(
    (BRATS / "slice_cache" / "val").glob("*.pt")
)

test_images = sorted(
    (BRATS / "test_volumes" / "images").glob("*.nii.gz")
)

test_labels = sorted(
    (BRATS / "test_volumes" / "labels").glob("*.nii.gz")
)

assert len(train_cache) == 388
assert len(val_cache) == 48
assert len(test_images) == 48
assert len(test_labels) == 48

for filename in [
    "resume_epoch150.pth",
    "best_through_epoch150.pth",
    "history_through_epoch150.csv",
    "part4_metadata.json",
]:
    path = PART4 / filename
    assert path.is_file(), path

    expected = expected_hash(
        part4_checksums,
        filename,
    )

    assert sha256(path) == expected, filename

metadata = json.loads(
    (PART4 / "part4_metadata.json").read_text()
)

assert metadata["next_epoch"] == 151
assert metadata["test_set_evaluated"] is False

print("Train patient caches:", len(train_cache))
print("Validation patient caches:", len(val_cache))
print("Held-out test patients:", len(test_images))
print("Part 4 continuation state: PASS")
print("Resume epoch: 150")


/tmp/ipykernel_23/3652696863.py:14: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(WORK)
/tmp/ipykernel_23/3652696863.py:20: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(WORK)


Train patient caches: 388
Validation patient caches: 48
Held-out test patients: 48
Part 4 continuation state: PASS
Resume epoch: 150


## Write the final experiment program

The architecture comes directly from the pinned author repository. The binary structure loss and the two tiny training utilities are reproduced locally with the same formulas/behavior, avoiding imports of unrelated author training entry points.

Before any long training starts, the notebook will:
- compile the program under Python 3.8,
- instantiate PVTv2-B2 + EMCAD,
- strictly load the epoch-150 checkpoint,
- verify parameter count,
- run a 352×352 forward pass,
- verify four one-channel outputs.


In [8]:
from pathlib import Path

training_code = '\nimport csv\nimport json\nimport logging\nimport math\nimport random\nimport time\nfrom datetime import datetime\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch.optim.lr_scheduler import CosineAnnealingLR\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision.transforms import InterpolationMode\nfrom torchvision.transforms import functional as TF\n\nfrom lib.networks import EMCADNet\n\n\n\n# =============================================================================\n# Author-equivalent binary loss and training utilities\n# =============================================================================\n# These definitions are copied behavior-for-behavior from the pinned EMCAD\n# training utilities. Keeping them local avoids importing unrelated dataloaders\n# and training entry points while preserving the same mathematics.\n\ndef structure_loss(pred, mask, w=1):\n    weit = 1 + 5 * torch.abs(\n        F.avg_pool2d(\n            mask,\n            kernel_size=31,\n            stride=1,\n            padding=15,\n        ) - mask\n    )\n\n    wbce = F.binary_cross_entropy_with_logits(\n        pred,\n        mask,\n        reduction="none",\n    )\n\n    wbce = (\n        (weit * wbce).sum(dim=(2, 3))\n        / weit.sum(dim=(2, 3))\n    )\n\n    pred = torch.sigmoid(pred)\n\n    inter = (\n        (pred * mask) * weit\n    ).sum(dim=(2, 3))\n\n    union = (\n        (pred + mask) * weit\n    ).sum(dim=(2, 3))\n\n    wiou = 1 - (\n        (inter + 1)\n        / (union - inter + 1)\n    )\n\n    return (\n        w * (wbce + wiou)\n    ).mean()\n\n\ndef clip_gradient(optimizer, grad_clip):\n    for group in optimizer.param_groups:\n        for param in group["params"]:\n            if param.grad is not None:\n                param.grad.data.clamp_(\n                    -grad_clip,\n                    grad_clip,\n                )\n\n\ndef adjust_lr(\n    optimizer,\n    init_lr,\n    epoch,\n    decay_rate=0.1,\n    decay_epoch=30,\n):\n    # This matches the author\'s utility. For our configured decay_epoch=300\n    # and epochs <=200 the multiplier is 1, so it is neutral.\n    decay = decay_rate ** (\n        epoch // decay_epoch\n    )\n\n    for param_group in optimizer.param_groups:\n        param_group["lr"] *= decay\n\n\n# =============================================================================\n# Configuration\n# =============================================================================\n\nWORK = Path("/kaggle/working")\nDATA_ROOT = WORK / "EMCAD_BraTS_Part1_Output"\nOUTPUT_ROOT = WORK / "EMCAD_BraTS_Run1_Part5"\n\nSEED = 2026\n\nSTART_EPOCH = 151\nEND_EPOCH = 200\nTOTAL_EPOCHS = 200\n\nIMG_SIZE = 352\nBATCH_SIZE = 8\nVAL_BATCH_SIZE = 16\n\nLR = 5e-4\nWEIGHT_DECAY = 1e-4\nETA_MIN = 1e-6\n\nCLIP = 0.5\nDECAY_RATE = 0.1\nDECAY_EPOCH = 300\n\nSIZE_RATES = [0.75, 1.0, 1.25]\nTHRESHOLD = 0.5\n\nKERNEL_SIZES = [1, 3, 5]\nEXPANSION_FACTOR = 2\nLGAG_KS = 3\nACTIVATION = "relu6"\nENCODER = "pvt_v2_b2"\nPRETRAINED_DIR = "./pretrained_pth/pvt/"\n\nAUTHOR_COMMIT = "26c9c31f731f749b62c5fe83f44376dac75f3aa8"\n\nIMAGENET_MEAN = torch.tensor(\n    [0.485, 0.456, 0.406]\n).view(3, 1, 1)\n\nIMAGENET_STD = torch.tensor(\n    [0.229, 0.224, 0.225]\n).view(3, 1, 1)\n\nOUTPUT_ROOT.mkdir(parents=True, exist_ok=True)\n\n\n# =============================================================================\n# Reproducibility\n# =============================================================================\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\n\n# =============================================================================\n# Prepared-data preflight\n# =============================================================================\n\ndef load_preparation_metadata():\n    split = json.loads(\n        (DATA_ROOT / "patient_split.json").read_text()\n    )\n\n    metadata = json.loads(\n        (DATA_ROOT / "preparation_metadata.json").read_text()\n    )\n\n    assert len(split["train"]) == 388\n    assert len(split["val"]) == 48\n    assert len(split["test"]) == 48\n\n    assert not (set(split["train"]) & set(split["val"]))\n    assert not (set(split["train"]) & set(split["test"]))\n    assert not (set(split["val"]) & set(split["test"]))\n\n    assert metadata["target"] == "Whole Tumour (WT): label > 0"\n\n    print("Patient split: 388 / 48 / 48")\n    print("Target:", metadata["target"])\n    print(\n        "Input modalities:",\n        metadata["selected_input_modalities"],\n    )\n    print("Patient leakage check: PASS")\n\n    return split, metadata\n\n\n# =============================================================================\n# PyTorch datasets\n# =============================================================================\n\ndef preprocess(image, mask, training):\n    image = image.float()\n    mask = mask.float()\n\n    if training:\n        if random.random() < 0.5:\n            angle = random.uniform(-90.0, 90.0)\n\n            image = TF.rotate(\n                image,\n                angle,\n                interpolation=InterpolationMode.BILINEAR,\n            )\n\n            mask = TF.rotate(\n                mask,\n                angle,\n                interpolation=InterpolationMode.NEAREST,\n            )\n\n        if random.random() < 0.5:\n            image = TF.hflip(image)\n            mask = TF.hflip(mask)\n\n        if random.random() < 0.5:\n            image = TF.vflip(image)\n            mask = TF.vflip(mask)\n\n    image = TF.resize(\n        image,\n        [IMG_SIZE, IMG_SIZE],\n        interpolation=InterpolationMode.BILINEAR,\n    )\n\n    mask = TF.resize(\n        mask,\n        [IMG_SIZE, IMG_SIZE],\n        interpolation=InterpolationMode.NEAREST,\n    )\n\n    image = (\n        image - IMAGENET_MEAN\n    ) / IMAGENET_STD\n\n    mask = (mask > 0.5).float()\n\n    return image, mask\n\n\nclass TrainPatientDataset(Dataset):\n    """\n    One training item = one patient.\n\n    Each epoch draws one slice at random from that patient\'s prepared\n    2D slice bank. This keeps patient representation balanced while\n    presenting different tumour/non-tumour slices across epochs.\n    """\n\n    def __init__(self):\n        paths = sorted(\n            (DATA_ROOT / "slice_cache" / "train").glob("*.pt")\n        )\n\n        if len(paths) != 388:\n            raise RuntimeError(\n                f"Expected 388 train patient caches, found {len(paths)}."\n            )\n\n        self.cases = [\n            torch.load(path, map_location="cpu")\n            for path in paths\n        ]\n\n    def __len__(self):\n        return len(self.cases)\n\n    def __getitem__(self, index):\n        case = self.cases[index]\n\n        sample_index = random.randrange(\n            case["images"].shape[0]\n        )\n\n        image = case["images"][sample_index]\n        mask = case["masks"][sample_index]\n\n        return preprocess(\n            image,\n            mask,\n            training=True,\n        )\n\n\nclass ValidationSliceDataset(Dataset):\n    """\n    Validation is deterministic: every cached validation slice is used.\n    """\n\n    def __init__(self):\n        paths = sorted(\n            (DATA_ROOT / "slice_cache" / "val").glob("*.pt")\n        )\n\n        if len(paths) != 48:\n            raise RuntimeError(\n                f"Expected 48 validation patient caches, found {len(paths)}."\n            )\n\n        self.samples = []\n\n        for path in paths:\n            case = torch.load(\n                path,\n                map_location="cpu",\n            )\n\n            for index in range(\n                case["images"].shape[0]\n            ):\n                self.samples.append(\n                    (\n                        case["patient_id"],\n                        int(\n                            case["slice_indices"][index].item()\n                        ),\n                        case["images"][index],\n                        case["masks"][index],\n                    )\n                )\n\n    def __len__(self):\n        return len(self.samples)\n\n    def __getitem__(self, index):\n        patient_id, slice_index, image, mask = (\n            self.samples[index]\n        )\n\n        image, mask = preprocess(\n            image,\n            mask,\n            training=False,\n        )\n\n        return (\n            image,\n            mask,\n            patient_id,\n            slice_index,\n        )\n\n\ndef build_loaders():\n    train_dataset = TrainPatientDataset()\n    val_dataset = ValidationSliceDataset()\n\n    shuffle_generator = torch.Generator()\n    shuffle_generator.manual_seed(SEED)\n\n    train_loader = DataLoader(\n        train_dataset,\n        batch_size=BATCH_SIZE,\n        shuffle=True,\n        num_workers=0,\n        pin_memory=True,\n        generator=shuffle_generator,\n    )\n\n    val_loader = DataLoader(\n        val_dataset,\n        batch_size=VAL_BATCH_SIZE,\n        shuffle=False,\n        num_workers=0,\n        pin_memory=True,\n    )\n\n    print("Train patients per epoch:", len(train_dataset))\n    print("Train batches per epoch:", len(train_loader))\n    print("Validation slices:", len(val_dataset))\n    print("Validation batches:", len(val_loader))\n\n    return train_loader, val_loader, shuffle_generator\n\n\n# =============================================================================\n# Model and author-style loss\n# =============================================================================\n\ndef build_model():\n    # Architecture is identical to the previous stages.\n    # pretrain=False only prevents an unnecessary attempt to read the original\n    # ImageNet initialization file. The complete epoch-150 model state is\n    # loaded with strict=True before training resumes.\n    model = EMCADNet(\n        num_classes=1,\n        kernel_sizes=KERNEL_SIZES,\n        expansion_factor=EXPANSION_FACTOR,\n        dw_parallel=True,\n        add=True,\n        lgag_ks=LGAG_KS,\n        activation=ACTIVATION,\n        encoder=ENCODER,\n        pretrain=False,\n        pretrained_dir=PRETRAINED_DIR,\n    )\n\n    return model\n\n\ndef emcad_loss(outputs, masks):\n    if not isinstance(outputs, list):\n        outputs = [outputs]\n\n    if len(outputs) != 4:\n        raise RuntimeError(\n            f"Expected four EMCAD outputs, got {len(outputs)}."\n        )\n\n    loss_p1 = structure_loss(outputs[0], masks)\n    loss_p2 = structure_loss(outputs[1], masks)\n    loss_p3 = structure_loss(outputs[2], masks)\n    loss_p4 = structure_loss(outputs[3], masks)\n\n    loss_p1234 = structure_loss(\n        outputs[0]\n        + outputs[1]\n        + outputs[2]\n        + outputs[3],\n        masks,\n    )\n\n    return (\n        loss_p1\n        + loss_p2\n        + loss_p3\n        + loss_p4\n        + loss_p1234\n    )\n\n\n# =============================================================================\n# Metrics\n# =============================================================================\n\ndef confusion_counts(prediction, target):\n    prediction = prediction.bool()\n    target = target.bool()\n\n    tp = torch.logical_and(\n        prediction,\n        target,\n    ).sum().item()\n\n    tn = torch.logical_and(\n        ~prediction,\n        ~target,\n    ).sum().item()\n\n    fp = torch.logical_and(\n        prediction,\n        ~target,\n    ).sum().item()\n\n    fn = torch.logical_and(\n        ~prediction,\n        target,\n    ).sum().item()\n\n    return tp, tn, fp, fn\n\n\ndef metrics_from_counts(tp, tn, fp, fn):\n    eps = 1e-6\n\n    return {\n        "dice": (\n            2 * tp + eps\n        ) / (\n            2 * tp + fp + fn + eps\n        ),\n        "iou": (\n            tp + eps\n        ) / (\n            tp + fp + fn + eps\n        ),\n        "sensitivity": (\n            tp + eps\n        ) / (\n            tp + fn + eps\n        ),\n        "specificity": (\n            tn + eps\n        ) / (\n            tn + fp + eps\n        ),\n        "precision": (\n            tp + eps\n        ) / (\n            tp + fp + eps\n        ),\n        "accuracy": (\n            tp + tn + eps\n        ) / (\n            tp + tn + fp + fn + eps\n        ),\n    }\n\n\ndef validate(model, loader, device):\n    model.eval()\n\n    tp = tn = fp = fn = 0\n\n    with torch.no_grad():\n        for images, masks, _, _ in loader:\n            images = images.to(\n                device,\n                non_blocking=True,\n            )\n\n            masks = masks.to(\n                device,\n                non_blocking=True,\n            ).float()\n\n            outputs = model(images)\n\n            if not isinstance(outputs, list):\n                outputs = [outputs]\n\n            logits = outputs[-1]\n\n            predictions = (\n                torch.sigmoid(logits)\n                >= THRESHOLD\n            )\n\n            targets = masks >= 0.5\n\n            counts = confusion_counts(\n                predictions,\n                targets,\n            )\n\n            tp += counts[0]\n            tn += counts[1]\n            fp += counts[2]\n            fn += counts[3]\n\n    return metrics_from_counts(\n        tp,\n        tn,\n        fp,\n        fn,\n    )\n\n\n# =============================================================================\n# Training\n# =============================================================================\n\ndef train_one_epoch(\n    model,\n    loader,\n    optimizer,\n    epoch,\n    device,\n):\n    model.train()\n\n    running_loss = 0.0\n    samples = 0\n\n    for step, (images, masks) in enumerate(\n        loader,\n        start=1,\n    ):\n        images = images.to(\n            device,\n            non_blocking=True,\n        )\n\n        masks = masks.to(\n            device,\n            non_blocking=True,\n        ).float()\n\n        for rate in SIZE_RATES:\n            optimizer.zero_grad()\n\n            if rate != 1.0:\n                train_size = int(\n                    round(\n                        IMG_SIZE * rate / 32\n                    ) * 32\n                )\n\n                current_images = F.interpolate(\n                    images,\n                    size=(train_size, train_size),\n                    mode="bilinear",\n                    align_corners=True,\n                )\n\n                current_masks = F.interpolate(\n                    masks,\n                    size=(train_size, train_size),\n                    mode="nearest",\n                )\n\n            else:\n                current_images = images\n                current_masks = masks\n\n            outputs = model(current_images)\n\n            loss = emcad_loss(\n                outputs,\n                current_masks,\n            )\n\n            loss.backward()\n\n            clip_gradient(\n                optimizer,\n                CLIP,\n            )\n\n            optimizer.step()\n\n            if rate == 1.0:\n                batch_size = images.size(0)\n\n                running_loss += (\n                    loss.item() * batch_size\n                )\n\n                samples += batch_size\n\n        if (\n            step % 25 == 0\n            or step == len(loader)\n        ):\n            print(\n                f"{datetime.now()} "\n                f"Epoch [{epoch:03d}/{TOTAL_EPOCHS:03d}], "\n                f"Step [{step:04d}/{len(loader):04d}], "\n                f"LR: {optimizer.param_groups[0][\'lr\']:.6f}, "\n                f"Loss: {running_loss / samples:.4f}"\n            )\n\n    return running_loss / samples\n\n\ndef save_history(path, history):\n    if not history:\n        return\n\n    with path.open("w", newline="") as f:\n        writer = csv.DictWriter(\n            f,\n            fieldnames=history[0].keys(),\n        )\n\n        writer.writeheader()\n        writer.writerows(history)\n\n\ndef save_resume_state(\n    path,\n    epoch,\n    model,\n    optimizer,\n    scheduler,\n    shuffle_generator,\n    best_val_dice,\n    best_epoch,\n    history,\n):\n    state = {\n        "stage": "BraTS Run 1 Part 5",\n        "epoch": epoch,\n        "total_epochs": TOTAL_EPOCHS,\n        "seed": SEED,\n        "author_commit": AUTHOR_COMMIT,\n        "model_state_dict": model.state_dict(),\n        "optimizer_state_dict": optimizer.state_dict(),\n        "scheduler_state_dict": scheduler.state_dict(),\n        "best_val_dice": best_val_dice,\n        "best_epoch": best_epoch,\n        "history": history,\n        "python_random_state": random.getstate(),\n        "numpy_random_state": np.random.get_state(),\n        "torch_rng_state": torch.get_rng_state(),\n        "cuda_rng_state_all": torch.cuda.get_rng_state_all(),\n        "shuffle_generator_state": shuffle_generator.get_state(),\n        "config": {\n            "img_size": IMG_SIZE,\n            "batch_size": BATCH_SIZE,\n            "lr": LR,\n            "weight_decay": WEIGHT_DECAY,\n            "eta_min": ETA_MIN,\n            "clip": CLIP,\n            "size_rates": SIZE_RATES,\n            "kernel_sizes": KERNEL_SIZES,\n            "expansion_factor": EXPANSION_FACTOR,\n            "lgag_ks": LGAG_KS,\n            "activation": ACTIVATION,\n            "encoder": ENCODER,\n        },\n    }\n\n    torch.save(\n        state,\n        path,\n    )\n\n\n# =============================================================================\n# Main\n# =============================================================================\n\n\n\n# =============================================================================\n# Exact continuation utilities\n# =============================================================================\n\ndef restore_rng_state(checkpoint, shuffle_generator):\n    random.setstate(checkpoint["python_random_state"])\n    np.random.set_state(checkpoint["numpy_random_state"])\n\n    torch.set_rng_state(\n        checkpoint["torch_rng_state"].cpu()\n    )\n\n    cuda_states = [\n        state.cpu()\n        for state in checkpoint["cuda_rng_state_all"]\n    ]\n\n    if not cuda_states:\n        raise RuntimeError(\n            "The resume checkpoint does not contain a CUDA RNG state."\n        )\n\n    # Training uses cuda:0 only. Restoring that exact saved generator state\n    # keeps the run continuous even if Kaggle exposes a different GPU count.\n    torch.cuda.set_rng_state(\n        cuda_states[0],\n        device=0,\n    )\n\n    shuffle_generator.set_state(\n        checkpoint["shuffle_generator_state"].cpu()\n    )\n\n\ndef move_optimizer_state_to_device(optimizer, device):\n    for state in optimizer.state.values():\n        for key, value in state.items():\n            if torch.is_tensor(value):\n                state[key] = value.to(device)\n\n\ndef verify_resume_checkpoint(checkpoint):\n    if checkpoint["epoch"] != 150:\n        raise RuntimeError(\n            "Expected the Part 4 resume checkpoint at epoch 150, "\n            f"found {checkpoint[\'epoch\']}."\n        )\n\n    if checkpoint["total_epochs"] != TOTAL_EPOCHS:\n        raise RuntimeError("Total epoch count changed.")\n\n    if checkpoint["seed"] != SEED:\n        raise RuntimeError("Run seed changed.")\n\n    if checkpoint["author_commit"] != AUTHOR_COMMIT:\n        raise RuntimeError("Author commit changed.")\n\n    expected_config = {\n        "img_size": IMG_SIZE,\n        "batch_size": BATCH_SIZE,\n        "lr": LR,\n        "weight_decay": WEIGHT_DECAY,\n        "eta_min": ETA_MIN,\n        "clip": CLIP,\n        "size_rates": SIZE_RATES,\n        "kernel_sizes": KERNEL_SIZES,\n        "expansion_factor": EXPANSION_FACTOR,\n        "lgag_ks": LGAG_KS,\n        "activation": ACTIVATION,\n        "encoder": ENCODER,\n    }\n\n    if checkpoint["config"] != expected_config:\n        raise RuntimeError(\n            "Training configuration differs from the previous stage."\n        )\n\n    history = checkpoint["history"]\n\n    if len(history) != 150:\n        raise RuntimeError(\n            f"Expected 150 cumulative history rows, found {len(history)}."\n        )\n\n    if int(history[-1]["epoch"]) != 150:\n        raise RuntimeError(\n            "Training history does not end at epoch 150."\n        )\n\n    print("Resume checkpoint verification: PASS")\n    print("Previous epoch:", checkpoint["epoch"])\n    print("Previous best epoch:", checkpoint["best_epoch"])\n    print(\n        "Previous best validation Dice:",\n        f"{checkpoint[\'best_val_dice\']:.4f}",\n    )\n\n\n# =============================================================================\n# Full-volume BraTS test utilities\n# =============================================================================\n\ndef case_id(path):\n    name = Path(path).name\n\n    if name.endswith(".nii.gz"):\n        return name[:-7]\n\n    return Path(name).stem\n\n\ndef write_rows_csv(path, rows):\n    if not rows:\n        return\n\n    with path.open("w", newline="") as f:\n        writer = csv.DictWriter(\n            f,\n            fieldnames=rows[0].keys(),\n        )\n\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef load_test_case(image_path, label_path):\n    import nibabel as nib\n\n    image_nii = nib.load(str(image_path))\n    label_nii = nib.load(str(label_path))\n\n    image_np = np.asarray(\n        image_nii.dataobj,\n        dtype=np.float32,\n    )\n\n    label_np = np.asarray(\n        label_nii.dataobj,\n        dtype=np.int16,\n    )\n\n    if image_np.ndim != 4:\n        raise RuntimeError(\n            f"Expected a 4D MRI volume, got {image_np.shape}: "\n            f"{image_path}"\n        )\n\n    modality_axes = [\n        axis\n        for axis, size in enumerate(image_np.shape)\n        if size == 4\n    ]\n\n    if len(modality_axes) != 1:\n        raise RuntimeError(\n            f"Cannot uniquely identify the MRI modality axis: "\n            f"{image_path} -> {image_np.shape}"\n        )\n\n    image_np = np.moveaxis(\n        image_np,\n        modality_axes[0],\n        0,\n    )\n\n    if image_np.shape[1:] != label_np.shape:\n        raise RuntimeError(\n            f"Image/label spatial mismatch for {image_path.name}: "\n            f"{image_np.shape} vs {label_np.shape}"\n        )\n\n    # dataset.json established the channel mapping:\n    # 0 = FLAIR, 2 = T1-Gd, 3 = T2.\n    selected = torch.from_numpy(\n        image_np[[0, 2, 3]]\n    ).float()\n\n    label = torch.from_numpy(\n        label_np\n    )\n\n    whole_tumour = (\n        label > 0\n    ).to(torch.uint8)\n\n    brain = (\n        selected.abs().sum(dim=0) > 0\n    )\n\n    normalized = torch.zeros_like(selected)\n\n    for channel in range(selected.shape[0]):\n        values = selected[channel][brain]\n\n        if values.numel() == 0:\n            raise RuntimeError(\n                f"Empty brain region: {image_path.name}"\n            )\n\n        mu = values.mean()\n        sigma = values.std()\n\n        if sigma <= 0:\n            raise RuntimeError(\n                f"Zero intensity variance: {image_path.name}, "\n                f"channel={channel}"\n            )\n\n        zscore = (\n            selected[channel] - mu\n        ) / sigma\n\n        zscore = zscore.clamp(\n            -5.0,\n            5.0,\n        )\n\n        normalized[channel][brain] = (\n            zscore[brain] + 5.0\n        ) / 10.0\n\n    return (\n        normalized,\n        whole_tumour,\n        brain,\n        image_nii.affine,\n        image_nii.header.copy(),\n    )\n\n\ndef preprocess_test_batch(batch, device):\n    batch = batch.to(\n        device,\n        non_blocking=True,\n    ).float()\n\n    batch = F.interpolate(\n        batch,\n        size=(IMG_SIZE, IMG_SIZE),\n        mode="bilinear",\n        align_corners=False,\n    )\n\n    mean_tensor = IMAGENET_MEAN.to(device)\n    std_tensor = IMAGENET_STD.to(device)\n\n    batch = (\n        batch - mean_tensor\n    ) / std_tensor\n\n    return batch\n\n\ndef hd95_surface_distance(prediction, target, spacing):\n    """\n    Symmetric 95th-percentile Hausdorff distance in physical units.\n\n    Returns NaN when either binary mask is empty. Empty-prediction cases are\n    reported explicitly rather than being hidden behind an arbitrary penalty.\n    """\n    from scipy.ndimage import (\n        binary_erosion,\n        distance_transform_edt,\n    )\n\n    prediction = prediction.astype(bool)\n    target = target.astype(bool)\n\n    if not prediction.any() or not target.any():\n        return float("nan")\n\n    pred_surface = np.logical_xor(\n        prediction,\n        binary_erosion(\n            prediction,\n            structure=np.ones((3, 3, 3), dtype=bool),\n            border_value=0,\n        ),\n    )\n\n    target_surface = np.logical_xor(\n        target,\n        binary_erosion(\n            target,\n            structure=np.ones((3, 3, 3), dtype=bool),\n            border_value=0,\n        ),\n    )\n\n    if not pred_surface.any() or not target_surface.any():\n        return float("nan")\n\n    distance_to_target = distance_transform_edt(\n        ~target_surface,\n        sampling=spacing,\n    )\n\n    distance_to_prediction = distance_transform_edt(\n        ~pred_surface,\n        sampling=spacing,\n    )\n\n    distances = np.concatenate(\n        [\n            distance_to_target[pred_surface],\n            distance_to_prediction[target_surface],\n        ]\n    )\n\n    return float(\n        np.percentile(\n            distances,\n            95,\n        )\n    )\n\n\ndef evaluate_full_test_case(\n    model,\n    image_path,\n    label_path,\n    device,\n    prediction_dir,\n    qualitative_dir,\n    save_qualitative=False,\n):\n    import nibabel as nib\n    from PIL import Image\n\n    (\n        image,\n        target,\n        brain,\n        affine,\n        header,\n    ) = load_test_case(\n        image_path,\n        label_path,\n    )\n\n    height, width, depth = target.shape\n\n    prediction = torch.zeros(\n        (height, width, depth),\n        dtype=torch.uint8,\n    )\n\n    # Processing only non-empty brain slices is equivalent to processing\n    # the complete volume because non-brain slices are explicitly kept at 0.\n    z_indices = [\n        z\n        for z in range(depth)\n        if brain[:, :, z].any().item()\n    ]\n\n    test_batch_size = 8\n\n    model.eval()\n\n    with torch.no_grad():\n        for start in range(\n            0,\n            len(z_indices),\n            test_batch_size,\n        ):\n            current_z = z_indices[\n                start:start + test_batch_size\n            ]\n\n            batch = torch.stack(\n                [\n                    image[:, :, :, z]\n                    for z in current_z\n                ],\n                dim=0,\n            )\n\n            batch = preprocess_test_batch(\n                batch,\n                device,\n            )\n\n            outputs = model(batch)\n\n            if not isinstance(outputs, list):\n                outputs = [outputs]\n\n            if len(outputs) != 4:\n                raise RuntimeError(\n                    f"Expected four EMCAD outputs, got {len(outputs)}."\n                )\n\n            # EMCAD paper: p4 is the final segmentation map.\n            logits = outputs[-1]\n\n            logits = F.interpolate(\n                logits,\n                size=(height, width),\n                mode="bilinear",\n                align_corners=False,\n            )\n\n            predicted = (\n                torch.sigmoid(logits)\n                >= THRESHOLD\n            ).to(\n                torch.uint8\n            ).cpu()\n\n            for local_index, z in enumerate(\n                current_z\n            ):\n                prediction[:, :, z] = (\n                    predicted[local_index, 0]\n                )\n\n    target_bool = target.bool()\n    prediction_bool = prediction.bool()\n\n    tp, tn, fp, fn = confusion_counts(\n        prediction_bool,\n        target_bool,\n    )\n\n    metrics = metrics_from_counts(\n        tp,\n        tn,\n        fp,\n        fn,\n    )\n\n    spacing = tuple(\n        float(value)\n        for value in header.get_zooms()[:3]\n    )\n\n    metrics["hd95"] = hd95_surface_distance(\n        prediction_bool.numpy(),\n        target_bool.numpy(),\n        spacing=spacing,\n    )\n\n    metrics["tp"] = int(tp)\n    metrics["tn"] = int(tn)\n    metrics["fp"] = int(fp)\n    metrics["fn"] = int(fn)\n\n    patient = case_id(label_path)\n\n    prediction_dir.mkdir(\n        parents=True,\n        exist_ok=True,\n    )\n\n    pred_nii = nib.Nifti1Image(\n        prediction.numpy().astype(np.uint8),\n        affine,\n        header=header,\n    )\n\n    prediction_path = (\n        prediction_dir\n        / f"{patient}_WT_pred.nii.gz"\n    )\n\n    nib.save(\n        pred_nii,\n        str(prediction_path),\n    )\n\n    if save_qualitative:\n        qualitative_dir.mkdir(\n            parents=True,\n            exist_ok=True,\n        )\n\n        tumour_area = target.sum(\n            dim=(0, 1)\n        )\n\n        z = int(\n            tumour_area.argmax().item()\n        )\n\n        flair = image[0, :, :, z].clone()\n        gt = target[:, :, z].float()\n        pred = prediction[:, :, z].float()\n\n        def to_uint8(tensor):\n            tensor = tensor.float()\n            low = tensor.min()\n            high = tensor.max()\n\n            if high > low:\n                tensor = (\n                    tensor - low\n                ) / (\n                    high - low\n                )\n\n            return (\n                tensor.clamp(0, 1) * 255\n            ).to(\n                torch.uint8\n            ).numpy()\n\n        Image.fromarray(\n            to_uint8(flair)\n        ).save(\n            qualitative_dir\n            / f"{patient}_z{z:03d}_FLAIR.png"\n        )\n\n        Image.fromarray(\n            to_uint8(gt)\n        ).save(\n            qualitative_dir\n            / f"{patient}_z{z:03d}_GT_WT.png"\n        )\n\n        Image.fromarray(\n            to_uint8(pred)\n        ).save(\n            qualitative_dir\n            / f"{patient}_z{z:03d}_PRED_WT.png"\n        )\n\n    return metrics\n\n\ndef evaluate_full_test_set(model, split, device):\n    image_root = (\n        DATA_ROOT\n        / "test_volumes"\n        / "images"\n    )\n\n    label_root = (\n        DATA_ROOT\n        / "test_volumes"\n        / "labels"\n    )\n\n    images = {\n        case_id(path): path\n        for path in image_root.glob("*.nii.gz")\n    }\n\n    labels = {\n        case_id(path): path\n        for path in label_root.glob("*.nii.gz")\n    }\n\n    test_patients = split["test"]\n\n    if set(test_patients) != set(images):\n        raise RuntimeError(\n            "Staged test image IDs do not match the fixed patient split."\n        )\n\n    if set(test_patients) != set(labels):\n        raise RuntimeError(\n            "Staged test label IDs do not match the fixed patient split."\n        )\n\n    prediction_dir = (\n        OUTPUT_ROOT\n        / "test_predictions"\n    )\n\n    qualitative_dir = (\n        OUTPUT_ROOT\n        / "qualitative_examples"\n    )\n\n    rows = []\n\n    global_tp = 0\n    global_tn = 0\n    global_fp = 0\n    global_fn = 0\n\n    for index, patient in enumerate(\n        test_patients,\n        start=1,\n    ):\n        metrics = evaluate_full_test_case(\n            model=model,\n            image_path=images[patient],\n            label_path=labels[patient],\n            device=device,\n            prediction_dir=prediction_dir,\n            qualitative_dir=qualitative_dir,\n            save_qualitative=(index <= 6),\n        )\n\n        row = {\n            "patient_id": patient,\n            "dice": metrics["dice"],\n            "iou": metrics["iou"],\n            "sensitivity": metrics["sensitivity"],\n            "specificity": metrics["specificity"],\n            "precision": metrics["precision"],\n            "accuracy": metrics["accuracy"],\n            "hd95": metrics["hd95"],\n            "tp": metrics["tp"],\n            "tn": metrics["tn"],\n            "fp": metrics["fp"],\n            "fn": metrics["fn"],\n        }\n\n        rows.append(row)\n\n        global_tp += metrics["tp"]\n        global_tn += metrics["tn"]\n        global_fp += metrics["fp"]\n        global_fn += metrics["fn"]\n\n        hd95_text = (\n            f"{metrics[\'hd95\']:.3f}"\n            if math.isfinite(metrics["hd95"])\n            else "NaN"\n        )\n\n        print(\n            f"Test {index:02d}/{len(test_patients):02d} "\n            f"{patient}: "\n            f"Dice={metrics[\'dice\']:.4f}, "\n            f"IoU={metrics[\'iou\']:.4f}, "\n            f"HD95={hd95_text}"\n        )\n\n    write_rows_csv(\n        OUTPUT_ROOT\n        / "test_patient_metrics.csv",\n        rows,\n    )\n\n    metric_names = [\n        "dice",\n        "iou",\n        "sensitivity",\n        "specificity",\n        "precision",\n        "accuracy",\n        "hd95",\n    ]\n\n    macro = {}\n\n    for name in metric_names:\n        values = [\n            float(row[name])\n            for row in rows\n            if math.isfinite(\n                float(row[name])\n            )\n        ]\n\n        macro[name] = {\n            "mean": float(\n                np.mean(values)\n            ),\n            "std": float(\n                np.std(\n                    values,\n                    ddof=1,\n                )\n            )\n            if len(values) > 1\n            else 0.0,\n            "n": len(values),\n        }\n\n    global_metrics = metrics_from_counts(\n        global_tp,\n        global_tn,\n        global_fp,\n        global_fn,\n    )\n\n    empty_predictions = sum(\n        1\n        for row in rows\n        if int(row["tp"]) + int(row["fp"]) == 0\n    )\n\n    summary = {\n        "patients": len(rows),\n        "macro_patient_metrics": macro,\n        "global_voxel_metrics": global_metrics,\n        "global_counts": {\n            "tp": int(global_tp),\n            "tn": int(global_tn),\n            "fp": int(global_fp),\n            "fn": int(global_fn),\n        },\n        "hd95_valid_patients": macro["hd95"]["n"],\n        "empty_prediction_patients": empty_predictions,\n        "threshold": THRESHOLD,\n        "test_selection_policy": (\n            "held-out patient split; evaluated once after 200 epochs "\n            "using the best validation checkpoint"\n        ),\n    }\n\n    (\n        OUTPUT_ROOT\n        / "test_summary.json"\n    ).write_text(\n        json.dumps(\n            summary,\n            indent=2,\n        )\n    )\n\n    return summary\n\n\n# =============================================================================\n# Final Run 1\n# =============================================================================\n\ndef main():\n    set_seed(SEED)\n\n    split, preparation_metadata = (\n        load_preparation_metadata()\n    )\n\n    device = torch.device(\n        "cuda"\n        if torch.cuda.is_available()\n        else "cpu"\n    )\n\n    if device.type != "cuda":\n        raise RuntimeError(\n            "GPU is required for training."\n        )\n\n    print("Device:", device)\n    print("GPU:", torch.cuda.get_device_name(0))\n    print("PyTorch:", torch.__version__)\n\n    train_loader, val_loader, shuffle_generator = (\n        build_loaders()\n    )\n\n    # Deterministic prepared-cache preflight.\n    images, masks = next(\n        iter(\n            DataLoader(\n                train_loader.dataset,\n                batch_size=2,\n                shuffle=False,\n                num_workers=0,\n            )\n        )\n    )\n\n    assert images.shape == (\n        2,\n        3,\n        IMG_SIZE,\n        IMG_SIZE,\n    )\n\n    assert masks.shape == (\n        2,\n        1,\n        IMG_SIZE,\n        IMG_SIZE,\n    )\n\n    assert masks.max().item() == 1.0\n\n    print("Prepared-cache preflight: PASS")\n\n    model = build_model().to(device)\n\n    optimizer = torch.optim.AdamW(\n        model.parameters(),\n        LR,\n        weight_decay=WEIGHT_DECAY,\n    )\n\n    scheduler = CosineAnnealingLR(\n        optimizer,\n        T_max=TOTAL_EPOCHS,\n        eta_min=ETA_MIN,\n    )\n\n    prior_root = (\n        WORK\n        / "EMCAD_BraTS_Run1_Part4_State"\n    )\n\n    resume_path = (\n        prior_root\n        / "resume_epoch150.pth"\n    )\n\n    prior_best_path = (\n        prior_root\n        / "best_through_epoch150.pth"\n    )\n\n    if not resume_path.is_file():\n        raise FileNotFoundError(\n            resume_path\n        )\n\n    if not prior_best_path.is_file():\n        raise FileNotFoundError(\n            prior_best_path\n        )\n\n    checkpoint = torch.load(\n        resume_path,\n        map_location=device,\n    )\n\n    verify_resume_checkpoint(\n        checkpoint\n    )\n\n    model.load_state_dict(\n        checkpoint["model_state_dict"],\n        strict=True,\n    )\n\n    optimizer.load_state_dict(\n        checkpoint["optimizer_state_dict"]\n    )\n\n    move_optimizer_state_to_device(\n        optimizer,\n        device,\n    )\n\n    scheduler.load_state_dict(\n        checkpoint["scheduler_state_dict"]\n    )\n\n    best_val_dice = float(\n        checkpoint["best_val_dice"]\n    )\n\n    best_epoch = int(\n        checkpoint["best_epoch"]\n    )\n\n    history = list(\n        checkpoint["history"]\n    )\n\n    best_path = (\n        OUTPUT_ROOT\n        / "best_through_epoch200.pth"\n    )\n\n    import shutil\n\n    shutil.copy2(\n        prior_best_path,\n        best_path,\n    )\n\n    # Restore the exact stochastic state only after all setup work.\n    restore_rng_state(\n        checkpoint,\n        shuffle_generator,\n    )\n\n    print(\n        "Resuming the same Run 1 at epoch:",\n        START_EPOCH,\n    )\n\n    print(\n        "Scheduler last_epoch:",\n        scheduler.last_epoch,\n    )\n\n    print(\n        "Optimizer LR at resume:",\n        f"{optimizer.param_groups[0][\'lr\']:.6f}",\n    )\n\n    log_path = (\n        OUTPUT_ROOT\n        / "training_part5.log"\n    )\n\n    logging.basicConfig(\n        filename=log_path,\n        level=logging.INFO,\n        format="[%(asctime)s] %(message)s",\n        force=True,\n    )\n\n    stage_start = time.time()\n\n    for epoch in range(\n        START_EPOCH,\n        END_EPOCH + 1,\n    ):\n        epoch_start = time.time()\n\n        adjust_lr(\n            optimizer,\n            LR,\n            epoch,\n            DECAY_RATE,\n            DECAY_EPOCH,\n        )\n\n        train_loss = train_one_epoch(\n            model=model,\n            loader=train_loader,\n            optimizer=optimizer,\n            epoch=epoch,\n            device=device,\n        )\n\n        val_metrics = validate(\n            model,\n            val_loader,\n            device,\n        )\n\n        epoch_time = (\n            time.time() - epoch_start\n        )\n\n        row = {\n            "epoch": epoch,\n            "lr": optimizer.param_groups[0]["lr"],\n            "train_loss": train_loss,\n            "val_dice": val_metrics["dice"],\n            "val_iou": val_metrics["iou"],\n            "val_sensitivity": val_metrics["sensitivity"],\n            "val_specificity": val_metrics["specificity"],\n            "val_precision": val_metrics["precision"],\n            "val_accuracy": val_metrics["accuracy"],\n            "epoch_time_seconds": epoch_time,\n        }\n\n        history.append(row)\n\n        message = (\n            f"Epoch: {epoch:03d}, Dataset: val, "\n            f"Dice: {val_metrics[\'dice\']:.4f}, "\n            f"IoU: {val_metrics[\'iou\']:.4f}, "\n            f"Time: {epoch_time:.1f}s"\n        )\n\n        print(message)\n        logging.info(message)\n\n        if (\n            val_metrics["dice"]\n            > best_val_dice\n        ):\n            previous = best_val_dice\n\n            best_val_dice = (\n                val_metrics["dice"]\n            )\n\n            best_epoch = epoch\n\n            torch.save(\n                model.state_dict(),\n                best_path,\n            )\n\n            best_message = (\n                "### Best Model Saved "\n                f"(Dice improved from "\n                f"{previous:.4f} to "\n                f"{best_val_dice:.4f}) ###"\n            )\n\n            print(best_message)\n            logging.info(best_message)\n\n        scheduler.step()\n\n        save_history(\n            OUTPUT_ROOT\n            / "history_through_epoch200.csv",\n            history,\n        )\n\n        save_resume_state(\n            OUTPUT_ROOT\n            / "resume_latest.pth",\n            epoch=epoch,\n            model=model,\n            optimizer=optimizer,\n            scheduler=scheduler,\n            shuffle_generator=shuffle_generator,\n            best_val_dice=best_val_dice,\n            best_epoch=best_epoch,\n            history=history,\n        )\n\n    training_elapsed = (\n        time.time() - stage_start\n    )\n\n    final_resume = (\n        OUTPUT_ROOT\n        / "resume_epoch200.pth"\n    )\n\n    save_resume_state(\n        final_resume,\n        epoch=END_EPOCH,\n        model=model,\n        optimizer=optimizer,\n        scheduler=scheduler,\n        shuffle_generator=shuffle_generator,\n        best_val_dice=best_val_dice,\n        best_epoch=best_epoch,\n        history=history,\n    )\n\n    if len(history) != 200:\n        raise RuntimeError(\n            f"Expected 200 cumulative history rows, "\n            f"found {len(history)}."\n        )\n\n    print()\n    print("=" * 72)\n    print("TRAINING COMPLETE — 200/200 EPOCHS")\n    print("=" * 72)\n    print("Best validation epoch:", best_epoch)\n    print(\n        "Best validation Dice:",\n        f"{best_val_dice:.4f}",\n    )\n    print(\n        "Part 5 training time:",\n        f"{training_elapsed / 3600:.2f} hours",\n    )\n    print("=" * 72)\n\n    # -----------------------------------------------------------------\n    # Test is touched only now, after all 200 epochs and model selection.\n    # -----------------------------------------------------------------\n\n    print()\n    print(\n        "Loading the best validation checkpoint for the "\n        "single final held-out test evaluation..."\n    )\n\n    model.load_state_dict(\n        torch.load(\n            best_path,\n            map_location=device,\n        ),\n        strict=True,\n    )\n\n    test_start = time.time()\n\n    test_summary = (\n        evaluate_full_test_set(\n            model=model,\n            split=split,\n            device=device,\n        )\n    )\n\n    test_elapsed = (\n        time.time() - test_start\n    )\n\n    final_summary = {\n        "experiment": (\n            "EMCAD Experiment 2 - BraTS Neuroimaging "\n            "Domain Generalization"\n        ),\n        "run": 1,\n        "seed": SEED,\n        "epochs": TOTAL_EPOCHS,\n        "best_validation_epoch": best_epoch,\n        "best_validation_dice": best_val_dice,\n        "test": test_summary,\n        "part5_training_seconds": training_elapsed,\n        "final_test_seconds": test_elapsed,\n        "test_evaluated_once": True,\n        "early_stopping": False,\n        "author_commit": AUTHOR_COMMIT,\n        "architecture": "PVTv2-B2 + EMCAD",\n        "runtime_bootstrap": "public pinned EMCAD commit; no Notebook-1 mount",\n        "target": "Whole Tumour (WT), binary",\n        "input_modalities": [\n            "FLAIR",\n            "T1-Gd",\n            "T2",\n        ],\n    }\n\n    (\n        OUTPUT_ROOT\n        / "final_summary.json"\n    ).write_text(\n        json.dumps(\n            final_summary,\n            indent=2,\n        )\n    )\n\n    part5_metadata = {\n        "experiment": (\n            "EMCAD Experiment 2 - BraTS Domain Generalization"\n        ),\n        "run": 1,\n        "stage": "Part 5 FINAL",\n        "epochs_this_stage": [151, 200],\n        "epochs_completed_total": [1, 200],\n        "total_epochs": TOTAL_EPOCHS,\n        "seed": SEED,\n        "resumed_from_epoch": 150,\n        "best_validation_epoch": best_epoch,\n        "best_validation_dice": best_val_dice,\n        "test_set_evaluated": True,\n        "test_evaluation_count": 1,\n        "early_stopping": False,\n        "author_commit": AUTHOR_COMMIT,\n    }\n\n    (\n        OUTPUT_ROOT\n        / "part5_metadata.json"\n    ).write_text(\n        json.dumps(\n            part5_metadata,\n            indent=2,\n        )\n    )\n\n    macro = (\n        test_summary[\n            "macro_patient_metrics"\n        ]\n    )\n\n    global_metrics = (\n        test_summary[\n            "global_voxel_metrics"\n        ]\n    )\n\n    print()\n    print("=" * 72)\n    print("FINAL RESULTS — BRATS RUN 1")\n    print("=" * 72)\n    print("Seed:", SEED)\n    print("Best epoch:", best_epoch)\n    print(\n        "Best validation Dice:",\n        f"{best_val_dice:.4f}",\n    )\n    print(\n        "Test Dice (patient macro):",\n        f"{macro[\'dice\'][\'mean\']:.4f} "\n        f"± {macro[\'dice\'][\'std\']:.4f}",\n    )\n    print(\n        "Test IoU (patient macro):",\n        f"{macro[\'iou\'][\'mean\']:.4f} "\n        f"± {macro[\'iou\'][\'std\']:.4f}",\n    )\n    print(\n        "Test Dice (global voxel):",\n        f"{global_metrics[\'dice\']:.4f}",\n    )\n    print(\n        "Test IoU (global voxel):",\n        f"{global_metrics[\'iou\']:.4f}",\n    )\n\n    if (\n        macro["hd95"]["n"] > 0\n    ):\n        print(\n            "Test HD95:",\n            f"{macro[\'hd95\'][\'mean\']:.3f} "\n            f"± {macro[\'hd95\'][\'std\']:.3f}",\n            f"(n={macro[\'hd95\'][\'n\']})",\n        )\n\n    print(\n        "Empty-prediction patients:",\n        test_summary[\n            "empty_prediction_patients"\n        ],\n    )\n\n    print(\n        "Final test time:",\n        f"{test_elapsed / 60:.1f} minutes",\n    )\n    print("=" * 72)\n    print("BRATS RUN 1 COMPLETE")\n\n\nif __name__ == "__main__":\n    main()\n'

script_path = REPO / "experiment2_brats_part5_final_v2.py"
script_path.write_text(training_code)
print("Final script:", script_path)
print("Lines:", len(training_code.splitlines()))


Final script: /kaggle/working/EMCAD/experiment2_brats_part5_final_v2.py
Lines: 1938


In [9]:
subprocess.run(
    [
        str(PYTHON),
        "-m",
        "py_compile",
        str(
            REPO
            / "experiment2_brats_part5_final_v2.py"
        ),
    ],
    cwd=REPO,
    check=True,
)

print("Python 3.8 syntax check: PASS")


Python 3.8 syntax check: PASS


In [10]:
# Fast architecture/checkpoint preflight before the 4-hour training stage.

preflight = r"""
import torch
from lib.networks import EMCADNet

checkpoint = torch.load(
    "/kaggle/working/EMCAD_BraTS_Run1_Part4_State/resume_epoch150.pth",
    map_location="cpu",
)

assert checkpoint["epoch"] == 150
assert checkpoint["total_epochs"] == 200
assert checkpoint["author_commit"] == "26c9c31f731f749b62c5fe83f44376dac75f3aa8"

model = EMCADNet(
    num_classes=1,
    kernel_sizes=[1, 3, 5],
    expansion_factor=2,
    dw_parallel=True,
    add=True,
    lgag_ks=3,
    activation="relu6",
    encoder="pvt_v2_b2",
    pretrain=False,
    pretrained_dir="./pretrained_pth/pvt/",
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

params = sum(p.numel() for p in model.parameters())
print("Parameters:", params)

# Do NOT hard-code an exact parameter total here.
# The authoritative compatibility check is the strict checkpoint load above:
# model.load_state_dict(..., strict=True). If any model key or tensor shape
# differs from the epoch-150 checkpoint, PyTorch raises an error immediately.
#
# The current pinned model reports 26,764,411 parameters in this runtime.
# We only retain a broad corruption sanity check so harmless bookkeeping/count
# differences cannot abort a valid continuation run.
assert 26_000_000 < params < 28_000_000, params

model_keys = set(model.state_dict().keys())
checkpoint_keys = set(checkpoint["model_state_dict"].keys())

assert model_keys == checkpoint_keys, (
    "Model/checkpoint state-dict keys differ."
)

for name, tensor in model.state_dict().items():
    saved = checkpoint["model_state_dict"][name]
    assert tuple(tensor.shape) == tuple(saved.shape), (
        name,
        tuple(tensor.shape),
        tuple(saved.shape),
    )

print("Checkpoint architecture compatibility: PASS")

device = torch.device("cuda:0")
model = model.to(device).eval()

with torch.no_grad():
    x = torch.zeros(
        1, 3, 352, 352,
        device=device,
    )
    outputs = model(x)

assert isinstance(outputs, list)
assert len(outputs) == 4

for i, output in enumerate(outputs, start=1):
    print(f"p{i}:", tuple(output.shape))
    assert output.shape == (1, 1, 352, 352)

print("Epoch-150 strict checkpoint load: PASS")
print("EMCAD forward preflight: PASS")
"""

preflight_path = REPO / "_part5_preflight.py"
preflight_path.write_text(preflight)

subprocess.run(
    [
        str(PYTHON),
        "-u",
        str(preflight_path),
    ],
    cwd=REPO,
    check=True,
)

print("LONG-RUN PREFLIGHT: PASS")


Model pvt_v2_b2 backbone:  created, param count: 24849856
Model EMCAD decoder:  created, param count: 1913515
Parameters: 26764411
Checkpoint architecture compatibility: PASS
p1: (1, 1, 352, 352)
p2: (1, 1, 352, 352)
p3: (1, 1, 352, 352)
p4: (1, 1, 352, 352)
Epoch-150 strict checkpoint load: PASS
EMCAD forward preflight: PASS
LONG-RUN PREFLIGHT: PASS


## Train epochs 151–200 and evaluate the held-out test set

Only run this cell after the previous cell prints:

`LONG-RUN PREFLIGHT: PASS`

The program resumes the exact epoch-150 state, finishes epoch 200, loads the best validation checkpoint, and then performs the single final evaluation on all 48 held-out 3D volumes.


In [11]:
stdout_path = (
    WORK
    / "EMCAD_BraTS_Run1_Part5_FINAL_v2_stdout.log"
)

with stdout_path.open("w") as log:
    process = subprocess.Popen(
        [
            str(PYTHON),
            "-u",
            "experiment2_brats_part5_final_v2.py",
        ],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()

    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Part 5 v2 failed with return code {return_code}. "
        f"See {stdout_path}"
    )

print("Part 5 FINAL v2 execution: PASS")


Patient split: 388 / 48 / 48
Target: Whole Tumour (WT): label > 0
Input modalities: {'FLAIR': 0, 'T1-Gd': 2, 'T2': 3}
Patient leakage check: PASS
Device: cuda
GPU: Tesla T4
PyTorch: 1.11.0+cu113
Train patients per epoch: 388
Train batches per epoch: 49
Validation slices: 480
Validation batches: 30
Prepared-cache preflight: PASS
Model pvt_v2_b2 backbone:  created, param count: 24849856
Model EMCAD decoder:  created, param count: 1913515
Resume checkpoint verification: PASS
Previous epoch: 150
Previous best epoch: 139
Previous best validation Dice: 0.8977
Resuming the same Run 1 at epoch: 151
Scheduler last_epoch: 150
Optimizer LR at resume: 0.000074
2026-10-06 18:20:46.125012 Epoch [151/200], Step [0025/0049], LR: 0.000074, Loss: 2.0437
2026-10-06 18:22:54.487345 Epoch [151/200], Step [0049/0049], LR: 0.000074, Loss: 1.9528
Epoch: 151, Dataset: val, Dice: 0.8944, IoU: 0.8089, Time: 280.3s
2026-10-06 18:25:32.335188 Epoch [152/200], Step [0025/0049], LR: 0.000071, Loss: 2.0808
2026-10-06

In [12]:
# Inspect and verify the final result before packaging.

RESULT = WORK / "EMCAD_BraTS_Run1_Part5"

summary = json.loads(
    (RESULT / "final_summary.json").read_text()
)

test = summary["test"]
macro = test["macro_patient_metrics"]
global_metrics = test["global_voxel_metrics"]

assert summary["epochs"] == 200
assert summary["test_evaluated_once"] is True
assert test["patients"] == 48
assert len(
    list(
        (RESULT / "test_predictions").glob("*.nii.gz")
    )
) == 48

print("FINAL BRATS RUN 1")
print("-" * 68)
print("Best epoch:", summary["best_validation_epoch"])
print(
    "Best validation Dice:",
    f"{summary['best_validation_dice']:.4f}",
)
print(
    "Test Dice (patient macro):",
    f"{macro['dice']['mean']:.4f}",
    "±",
    f"{macro['dice']['std']:.4f}",
)
print(
    "Test IoU (patient macro):",
    f"{macro['iou']['mean']:.4f}",
    "±",
    f"{macro['iou']['std']:.4f}",
)
print(
    "Test Dice (global voxel):",
    f"{global_metrics['dice']:.4f}",
)
print(
    "Test IoU (global voxel):",
    f"{global_metrics['iou']:.4f}",
)
print(
    "HD95:",
    f"{macro['hd95']['mean']:.3f}",
    "±",
    f"{macro['hd95']['std']:.3f}",
    f"(valid n={macro['hd95']['n']})",
)
print(
    "Empty-prediction patients:",
    test["empty_prediction_patients"],
)
print("Test patients:", test["patients"])
print()
print("FINAL RESULT VERIFICATION: PASS")


FINAL BRATS RUN 1
--------------------------------------------------------------------
Best epoch: 139
Best validation Dice: 0.8977
Test Dice (patient macro): 0.8613 ± 0.1079
Test IoU (patient macro): 0.7700 ± 0.1469
Test Dice (global voxel): 0.8984
Test IoU (global voxel): 0.8155
HD95: 16.862 ± 21.851 (valid n=48)
Empty-prediction patients: 0
Test patients: 48

FINAL RESULT VERIFICATION: PASS


## Package final evidence

This produces:
- compact evidence ZIP,
- final best/resume checkpoints,
- 48 volumetric NIfTI predictions,
- SHA256 manifest,
- full console log,
- exact final script and runtime manifest.


In [13]:
RESULT = WORK / "EMCAD_BraTS_Run1_Part5"

evidence_root = (
    WORK
    / "EMCAD_BraTS_Run1_FINAL_Evidence_v2"
)

if evidence_root.exists():
    shutil.rmtree(evidence_root)

evidence_root.mkdir()

for filename in [
    "final_summary.json",
    "test_summary.json",
    "test_patient_metrics.csv",
    "history_through_epoch200.csv",
    "part5_metadata.json",
    "training_part5.log",
]:
    source = RESULT / filename
    assert source.is_file(), source
    shutil.copy2(
        source,
        evidence_root / filename,
    )

for filename in [
    "patient_split.json",
    "preparation_metadata.json",
    "patient_audit.csv",
    "slice_cache_manifest.csv",
    "test_volume_manifest.csv",
]:
    source = BRATS / filename
    assert source.is_file(), source
    shutil.copy2(
        source,
        evidence_root / filename,
    )

qualitative_source = (
    RESULT / "qualitative_examples"
)

if qualitative_source.is_dir():
    shutil.copytree(
        qualitative_source,
        evidence_root / "qualitative_examples",
    )

shutil.copy2(
    REPO / "experiment2_brats_part5_final_v2.py",
    evidence_root / "experiment2_brats_part5_final_v2.py",
)

shutil.copy2(
    stdout_path,
    evidence_root / "full_stdout.log",
)

(evidence_root / "AUTHOR_COMMIT.txt").write_text(
    EXPECTED_COMMIT + "\n"
)

runtime_manifest = subprocess.check_output(
    [
        str(PIP),
        "freeze",
    ],
    text=True,
)

(evidence_root / "RUNTIME_PIP_FREEZE.txt").write_text(
    runtime_manifest
)

evidence_zip = shutil.make_archive(
    str(
        WORK
        / "EMCAD_BraTS_Run1_FINAL_Evidence_v2"
    ),
    "zip",
    root_dir=WORK,
    base_dir=evidence_root.name,
)

checkpoint_root = (
    WORK
    / "EMCAD_BraTS_Run1_FINAL_Checkpoints_v2"
)

if checkpoint_root.exists():
    shutil.rmtree(checkpoint_root)

checkpoint_root.mkdir()

for filename in [
    "best_through_epoch200.pth",
    "resume_epoch200.pth",
]:
    source = RESULT / filename
    assert source.is_file(), source
    shutil.copy2(
        source,
        checkpoint_root / filename,
    )

checkpoint_archive = shutil.make_archive(
    str(
        WORK
        / "EMCAD_BraTS_Run1_FINAL_Checkpoints_v2"
    ),
    "gztar",
    root_dir=WORK,
    base_dir=checkpoint_root.name,
)

prediction_dir = (
    RESULT / "test_predictions"
)

assert len(
    list(
        prediction_dir.glob("*.nii.gz")
    )
) == 48

prediction_archive = shutil.make_archive(
    str(
        WORK
        / "EMCAD_BraTS_Run1_FINAL_Predictions_v2"
    ),
    "gztar",
    root_dir=RESULT,
    base_dir="test_predictions",
)

checksum_path = (
    WORK
    / "EMCAD_BraTS_Run1_FINAL_v2_SHA256SUMS.txt"
)

with checksum_path.open("w") as f:
    for path in [
        Path(evidence_zip),
        Path(checkpoint_archive),
        Path(prediction_archive),
    ]:
        f.write(
            f"{sha256(path)}  {path.name}\n"
        )

print("Evidence:", evidence_zip)
print("Checkpoints:", checkpoint_archive)
print("Predictions:", prediction_archive)
print("Checksums:", checksum_path)


Evidence: /kaggle/working/EMCAD_BraTS_Run1_FINAL_Evidence_v2.zip
Checkpoints: /kaggle/working/EMCAD_BraTS_Run1_FINAL_Checkpoints_v2.tar.gz
Predictions: /kaggle/working/EMCAD_BraTS_Run1_FINAL_Predictions_v2.tar.gz
Checksums: /kaggle/working/EMCAD_BraTS_Run1_FINAL_v2_SHA256SUMS.txt


In [14]:
required_outputs = [
    WORK / "EMCAD_BraTS_Run1_FINAL_Evidence_v2.zip",
    WORK / "EMCAD_BraTS_Run1_FINAL_Checkpoints_v2.tar.gz",
    WORK / "EMCAD_BraTS_Run1_FINAL_Predictions_v2.tar.gz",
    WORK / "EMCAD_BraTS_Run1_FINAL_v2_SHA256SUMS.txt",
    WORK / "EMCAD_BraTS_Run1_Part5_FINAL_v2_stdout.log",
]

for path in required_outputs:
    assert path.is_file(), path
    print(
        path.name,
        f"{path.stat().st_size / 1024**2:.2f} MiB",
    )

commit = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

assert commit == EXPECTED_COMMIT

subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "diff",
        "--exit-code",
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "diff",
        "--cached",
        "--exit-code",
    ],
    check=True,
)

print()
print("=" * 72)
print("BRATS EXPERIMENT 2 — RUN 1 FINAL v2 COMPLETE")
print("=" * 72)
print("Training: 200 / 200 epochs")
print("Held-out test patients: 48")
print("Notebook-1 mount dependency: NONE")
print("Pinned author source: CLEAN")
print("Save this Kaggle notebook as a Version with outputs.")


EMCAD_BraTS_Run1_FINAL_Evidence_v2.zip 0.12 MiB
EMCAD_BraTS_Run1_FINAL_Checkpoints_v2.tar.gz 381.91 MiB
EMCAD_BraTS_Run1_FINAL_Predictions_v2.tar.gz 2.41 MiB
EMCAD_BraTS_Run1_FINAL_v2_SHA256SUMS.txt 0.00 MiB
EMCAD_BraTS_Run1_Part5_FINAL_v2_stdout.log 0.02 MiB

BRATS EXPERIMENT 2 — RUN 1 FINAL v2 COMPLETE
Training: 200 / 200 epochs
Held-out test patients: 48
Notebook-1 mount dependency: NONE
Pinned author source: CLEAN
Save this Kaggle notebook as a Version with outputs.
